# Bus Booking Analytics - ETL Pipeline

This notebook implements the complete **ETL (Extract, Transform, Load)** pipeline as described in the project documentation.
It covers:
1. **Extract**: Loading raw dirty CSV datasets (`bus_booking_raw.csv`, `customers.csv`, `buses.csv`, `routes.csv`).
2. **Transform & Validate**:
   - Flexible Date Parsing using `format='mixed'` to resolve inconsistent date formats.
   - Deduplication (based on `Booking_ID`)
   - Handling Null Values (e.g. dropping bookings with null fares, filling missing customer phone numbers with 'Unknown')
   - Data Standardization (dates formatted to YYYY-MM-DD, proper title casing for names and routes)
   - Data Consistency Checks (Travel Date >= Booking Date, Fare > 0, Seat Number within physical Bus Capacity)
   - Referential Integrity checks between all foreign keys.
3. **Load**: Loading validated data into local **SQLite** database and outlining connection to **MySQL** database.

### Step 1: Extract (Load Raw CSVs)

In [1]:
import pandas as pd
import os
import sqlite3
import sys

# Import etl_pipeline module from current directory
import etl_pipeline

data_dir = '../data'
bookings, customers, buses, routes = etl_pipeline.load_csv_data(data_dir)
print(f"Raw Bookings count: {len(bookings)}")
print(f"Raw Customers count: {len(customers)}")

Loading CSV files...
Loading raw bookings from ../data\bus_booking_raw.csv
Raw Bookings count: 2100
Raw Customers count: 301


### Step 2: Transform, Clean & Validate
We run the validation logic that details formatting repairs, drop counts, and deduplications.

In [2]:
bookings_clean, customers_clean, buses_clean, routes_clean = etl_pipeline.transform_and_validate(
    bookings, customers, buses, routes
)

--- ETL CLEANING & TRANSFORMATION REPORT ---
Deduplication: Removed 100 duplicate booking records.
Null Handling: Removed 39 booking records with missing Fare_Amount.
Null Handling: Filled 21 missing customer phone numbers with 'Unknown'.
Null Handling/Standardization: Cleansed Gender field, filling 9 missing values.
Null Handling/Outliers: Cleansed Age field, filling 16 nulls and correcting 30 outliers to median (48).
Constraint Check: Removed 38 bookings where Travel_Date was before Booking_Date.
Constraint Check: Removed 29 bookings with non-positive Fare_Amount (e.g. negative values).
Capacity Check: Removed 20 bookings where Seat_Number exceeded Bus Capacity.

--- SUMMARY OF PROCESS ---
Bookings: 2100 raw rows -> 1874 clean rows (Removed 226 rows).
Customers: 301 raw rows -> 301 clean rows.
--------------------------------------------


### Step 3: Load into local SQLite Database

In [3]:
db_path = '../data/bus_booking_analytics.db'
etl_pipeline.load_to_sqlite(bookings_clean, customers_clean, buses_clean, routes_clean, db_path)

Loading data into SQLite database: ../data/bus_booking_analytics.db...


Successfully loaded datasets into SQLite database.


### Step 4: Verification Queries via SQL
Let's run queries on our cleaned dataset in the SQLite database to generate Key Business Metrics (Section 9).

In [4]:
conn = sqlite3.connect(db_path)

print("--- 1. KEY METRICS SUMMARY ---")
metrics_query = """
SELECT 
    COUNT(Booking_ID) as Total_Bookings,
    ROUND(SUM(Fare_Amount), 2) as Total_Revenue,
    ROUND(AVG(Fare_Amount), 2) as Avg_Fare
FROM Bookings
"""
print(pd.read_sql_query(metrics_query, conn))

print("\n--- 2. TOP 5 ROUTES BY REVENUE ---")
routes_query = """
SELECT 
    r.Source, 
    r.Destination, 
    COUNT(b.Booking_ID) as Booking_Count,
    SUM(b.Fare_Amount) as Route_Revenue
FROM Bookings b
JOIN Routes r ON b.Route_ID = r.Route_ID
GROUP BY r.Route_ID
ORDER BY Route_Revenue DESC
LIMIT 5;
"""
print(pd.read_sql_query(routes_query, conn))

print("\n--- 3. REVENUE BY BUS TYPE ---")
bus_query = """
SELECT 
    bus.Bus_Type, 
    COUNT(b.Booking_ID) as Booking_Count,
    SUM(b.Fare_Amount) as Revenue
FROM Bookings b
JOIN Buses bus ON b.Bus_ID = bus.Bus_ID
GROUP BY bus.Bus_Type
ORDER BY Revenue DESC;
"""
print(pd.read_sql_query(bus_query, conn))

print("\n--- 4. OCCUPANCY RATE ESTIMATE PER BUS ---")
occupancy_query = """
SELECT 
    b.Bus_ID,
    bus.Bus_Number,
    bus.Capacity,
    COUNT(b.Booking_ID) as Bookings_Made,
    ROUND((COUNT(b.Booking_ID) * 100.0) / (bus.Capacity * 20.0), 2) as Estimated_Occupancy_Percentage
FROM Bookings b
JOIN Buses bus ON b.Bus_ID = bus.Bus_ID
GROUP BY b.Bus_ID
ORDER BY Estimated_Occupancy_Percentage DESC;
"""
print(pd.read_sql_query(occupancy_query, conn))

conn.close()

--- 1. KEY METRICS SUMMARY ---
   Total_Bookings  Total_Revenue  Avg_Fare
0            1874      1469440.0    784.12

--- 2. TOP 5 ROUTES BY REVENUE ---
      Source Destination  Booking_Count  Route_Revenue
0    Chennai   Hyderabad            165       182475.0
1    Kolkata       Patna            177       182040.0
2  Bangalore   Hyderabad            176       178030.0
3     Mumbai   Ahmedabad            185       174400.0
4  Hyderabad   Bangalore            165       166875.0

--- 3. REVENUE BY BUS TYPE ---
         Bus_Type  Booking_Count   Revenue
0      AC Sleeper            676  561695.0
1  Non-AC Sleeper            664  550460.0
2       AC Seater            534  357285.0

--- 4. OCCUPANCY RATE ESTIMATE PER BUS ---
    Bus_ID    Bus_Number  Capacity  Bookings_Made  \
0      201  DL-95-B-9935        40            186   
1      202  GJ-05-A-2535        40            179   
2      206  DL-28-F-2674        40            177   
3      204  GJ-29-H-5557        40            167   
4   

### Step 5: Loading into MySQL (MySQL Database Setup)
Configure your host, user, and password below and run the function to load clean data into MySQL.

In [5]:
mysql_config = {
    'host': 'localhost',
    'user': 'root',
    'password': 'root',
    'database': 'bus_booking_analytics'
}

# Uncomment below to run if you have MySQL active on your machine:
# etl_pipeline.load_to_mysql(bookings_clean, customers_clean, buses_clean, routes_clean, mysql_config)